# BiLSTM + CRF for Token Classification (NER) — Raw PyTorch

This replaces the earlier BERT+CRF attempt, which got stuck at low F1 despite extensive
debugging — likely because BERT (110M params) and a freshly-initialized CRF transition
matrix (~90 params) converge at very different speeds, and 3 epochs wasn't enough for
both to settle together.

BiLSTM+CRF is the **classic, pre-BERT NER architecture**. It's much smaller, trains fast,
and the CRF actually gets to learn real transition structure within a few epochs because
there's no huge pretrained encoder dominating the gradient dynamics.

Compare against `bert_ner_trainer.ipynb` (HF Trainer, BERT + plain softmax, F1 ~0.91-0.95).


In [ ]:
!pip install -q torch datasets seqeval evaluate pytorch-crf tqdm

## 1. Config

In [ ]:
import torch

DATASET_NAME = "tomaarsen/conll2003"
OUTPUT_DIR = "./bilstm-crf"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
EPOCHS = 15          # BiLSTM+CRF needs more epochs than BERT, but each epoch is much faster
EMBEDDING_DIM = 100
HIDDEN_DIM = 256      # BiLSTM hidden size (output will be 2x this, for both directions)
LR = 1e-3            # much higher than BERT fine-tuning LR -- this is training from scratch

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(DEVICE)

## 2. Load data

Same CoNLL-2003 dataset as before. This time we build our **own vocabulary** from the
training set, since there's no pretrained tokenizer involved — BiLSTM-CRF learns its own
word embeddings from scratch (or you could plug in pretrained GloVe vectors, but we keep
it simple here).

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset(DATASET_NAME)
label_names = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
num_labels = len(label_names)
print(label_names)

## 3. Build vocabulary

Map every word in the training set to an integer id. Unknown words at eval/test time map
to a special `<UNK>` token; padding uses `<PAD>` (id 0).

In [ ]:
from collections import Counter

PAD_IDX = 0
UNK_IDX = 1

word_counter = Counter()
for example in raw_datasets["train"]:
    word_counter.update(w.lower() for w in example["tokens"])

# keep all words that appear at least once (CoNLL-2003 is small enough that this is fine)
vocab = ["<PAD>", "<UNK>"] + sorted(word_counter.keys())
word2id = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
print("vocab size:", vocab_size)

## 4. Tokenize (word -> id) and build the Dataset/DataLoader

No subword splitting here — one id per word, one label per word. This sidesteps the
entire `[CLS]`/subword-alignment problem that caused issues in the BERT+CRF version.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class NERDataset(Dataset):
    def __init__(self, hf_split):
        self.examples = hf_split

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]
        word_ids = [word2id.get(w.lower(), UNK_IDX) for w in example["tokens"]]
        labels = example["ner_tags"]
        return {"word_ids": word_ids, "labels": labels}


def collate_fn(batch):
    max_len = max(len(x["word_ids"]) for x in batch)
    word_ids = torch.full((len(batch), max_len), PAD_IDX, dtype=torch.long)
    labels = torch.zeros((len(batch), max_len), dtype=torch.long)
    mask = torch.zeros((len(batch), max_len), dtype=torch.bool)

    for i, x in enumerate(batch):
        seq_len = len(x["word_ids"])
        word_ids[i, :seq_len] = torch.tensor(x["word_ids"], dtype=torch.long)
        labels[i, :seq_len] = torch.tensor(x["labels"], dtype=torch.long)
        mask[i, :seq_len] = True

    return {"word_ids": word_ids, "labels": labels, "mask": mask}


train_loader = DataLoader(NERDataset(raw_datasets["train"]), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
eval_loader = DataLoader(NERDataset(raw_datasets["validation"]), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(NERDataset(raw_datasets["test"]), batch_size=BATCH_SIZE, collate_fn=collate_fn)

# Sanity check: mask's first column must be all True for the CRF library (true here,
# since every real sentence has at least 1 word -- no [CLS]-style special token to skip)
batch = next(iter(train_loader))
print("mask[:, 0] all True:", batch["mask"][:, 0].all().item())

## 5. Model: word embeddings + BiLSTM + linear emissions + CRF

No subword/[CLS] complications — every position in the sequence is a real word, so the
mask's first column is naturally all `True`, which is exactly what `torchcrf` requires.

In [ ]:
import torch.nn as nn
from torchcrf import CRF

class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers=1,
            bidirectional=True, batch_first=True,
        )
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)  # *2 for bidirectional
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, word_ids, labels=None, mask=None):
        embedded = self.embedding(word_ids)
        lstm_out, _ = self.lstm(embedded)
        emissions = self.classifier(self.dropout(lstm_out))

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)


model = BiLSTMCRF(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, num_labels).to(DEVICE)
print(model)
print("total params:", sum(p.numel() for p in model.parameters()))

## 6. Optimizer / scheduler

Everything here trains from scratch (no pretrained weights), so a single learning rate
across all parameters is fine — there's no "pretrained vs randomly-initialized" speed
mismatch like there was with BERT+CRF.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

## 7. Evaluation loop

In [ ]:
import evaluate

seqeval = evaluate.load("seqeval")

def evaluate_loop(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            word_ids = batch["word_ids"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            labels = batch["labels"]

            decoded = model(word_ids, mask=mask)  # list of lists, variable length

            for i, seq in enumerate(decoded):
                mask_i = mask[i].cpu()
                true_seq = labels[i][mask_i].tolist()
                all_preds.append([id2label[p] for p in seq])
                all_labels.append([id2label[l] for l in true_seq])

    results = seqeval.compute(predictions=all_preds, references=all_labels)
    return results["overall_f1"], results

## 8. Training loop

In [ ]:
from tqdm.auto import tqdm

def train():
    best_f1 = -1.0
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
        for batch in progress:
            word_ids = batch["word_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)

            optimizer.zero_grad()
            loss = model(word_ids, labels=labels, mask=mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        avg_loss = total_loss / len(train_loader)
        val_f1, _ = evaluate_loop(eval_loader)
        print(f"Epoch {epoch + 1}: train_loss={avg_loss:.4f}  val_f1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")

    model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pt"))
    test_f1, test_results = evaluate_loop(test_loader)
    print(f"Test F1: {test_f1:.4f}")
    print(test_results)

train()

## What changed vs. the BERT+CRF attempt

- **No subword tokenization** — one id per word, one label per word. This removes the
  entire `[CLS]`/subword-alignment complexity that the BERT version needed (no slicing,
  no first-subword masking logic).
- **No pretrained weights** — embeddings and the LSTM train from scratch, so there's no
  speed mismatch between "pretrained, needs small updates" and "randomly initialized,
  needs large updates." A single learning rate works for the whole model.
- **Far fewer parameters overall** (embeddings + a 1-layer BiLSTM + a small classifier +
  the CRF) — easier for the CRF's tiny transition matrix to actually influence training,
  since it's not competing against 110M BERT parameters for gradient signal.
- **More epochs (15 vs 3)** — appropriate here since each epoch is much faster without
  BERT's forward/backward cost, and training-from-scratch generally needs more epochs
  than fine-tuning a pretrained model.

If you want to push accuracy further, the standard next steps are: pretrained word
embeddings (GloVe/FastText) instead of training embeddings from scratch, adding character-level
features (a small char-CNN or char-LSTM, which is what the original BiLSTM-CRF NER papers
used to handle unknown words and morphology), or stacking 2 BiLSTM layers instead of 1.
